
# Target Inference Notebook

- 입력: `./dataset/train.csv`, `./dataset/TEST_*.csv`
- 기본 동작: 저장된 TEST 예측 결과를 모아 **submission** 형태로 변환
- 옵션: PatchTST 체크포인트가 있으면 직접 추론 가능
- 출력: `./result/submission_patchtst.csv` (또는 `submission_from_saved_preds.csv`)


In [3]:

# %% [Setup] Imports and paths
import os, sys, glob, json, math, warnings
from datetime import datetime, timedelta
import pandas as pd
import numpy as np

warnings.filterwarnings("ignore")

# Paths (edit if your repo uses different layout)
DATA_DIR = "./dataset_new"
RESULT_DIR = "./result"
ARTIFACT_DIR = "./artifacts_patchtst"  # where ckpt/config may live
PRED_DIRS_CANDIDATES = [
    os.path.join(RESULT_DIR, "preds"),
    os.path.join(RESULT_DIR, "test_preds"),
    RESULT_DIR,
]

# Constants
REQUIRED_COLS = ["date", "store", "menu", "store_menu", "sales", "date_ordinal"]
L, H = 28, 7  # lookback, horizon

os.makedirs(RESULT_DIR, exist_ok=True)

# Utility: assert required columns
def assert_required_cols(df, cols=REQUIRED_COLS, name="df"):
    missing = [c for c in cols if c not in df.columns]
    assert not missing, f"[{name}] missing columns: {missing}"

# Utility: safe sMAPE for diagnostics
def smape(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.maximum(np.abs(y_true)+np.abs(y_pred), eps)
    return 200.0*np.mean(np.abs(y_pred - y_true)/denom)

print("Environment ready.")


Environment ready.


In [4]:

# %% [Data] Load train and TEST_* meta
train_path = os.path.join(DATA_DIR, "train.csv")
assert os.path.exists(train_path), f"train.csv not found at {train_path}"
train = pd.read_csv(train_path)
assert_required_cols(train, name="train")

# TEST files
test_paths = sorted(glob.glob(os.path.join(DATA_DIR, "TEST_*.csv")))
assert test_paths, f"No TEST_*.csv found under {DATA_DIR}"
tests = {os.path.splitext(os.path.basename(p))[0]: pd.read_csv(p) for p in test_paths}
for k,df in tests.items():
    assert_required_cols(df, name=k)
    # Verify each TEST file has 28 days
    _n_days = df['date'].nunique()
    assert _n_days == L, f"{k}: expected {L} days, found { _n_days }"

print(f"Loaded train ({len(train)} rows) and {len(tests)} TEST files.")
print(list(tests.keys())[:3], '...')


AssertionError: [train] missing columns: ['date_ordinal']

In [ ]:

# %% [Dates] Build future target date index per TEST file
def parse_date(d):
    # handle either string or int ordinal
    if isinstance(d, str):
        return pd.to_datetime(d)
    return pd.to_datetime(datetime.fromordinal(int(d)).date())

def next_horizon_dates(last_date_str, H=7):
    last_dt = pd.to_datetime(last_date_str)
    return [(last_dt + pd.Timedelta(days=i)).strftime("%Y-%m-%d") for i in range(1, H+1)]

future_dates_map = {}
for k, df in tests.items():
    last_date = pd.to_datetime(df["date"]).max().strftime("%Y-%m-%d")
    future_dates_map[k] = next_horizon_dates(last_date, H=H)

# Sanity check: show one example
ex_k = sorted(future_dates_map.keys())[0]
print(ex_k, "->", future_dates_map[ex_k])


In [ ]:

# %% [Mode A] Assemble submission from saved TEST predictions (preferred if you already ran inference per TEST window)

def find_saved_pred_files():
    hits = []
    for d in PRED_DIRS_CANDIDATES:
        if not os.path.isdir(d):
            continue
        # common patterns: one file per TEST, or a single combined file
        patterns = [
            os.path.join(d, "TEST_*.csv"),
            os.path.join(d, "pred_TEST_*.csv"),
            os.path.join(d, "test_preds_*.csv"),
            os.path.join(d, "predictions_*.csv"),
            os.path.join(d, "all_test_predictions.csv"),
        ]
        for pat in patterns:
            hits.extend(glob.glob(pat))
    return sorted(set(hits))

saved_pred_files = find_saved_pred_files()
print("Found saved pred files:", saved_pred_files)

def load_flexible_pred(fp):
    # Load a prediction file with flexible schema.
    # Acceptable forms:
    #   (1) long: [store_menu, date, pred]
    #   (2) wide: [store_menu, pred_d1, ..., pred_d7] plus optional TEST id
    df = pd.read_csv(fp)
    # Try to detect schema
    cols = [c.lower() for c in df.columns]
    lcmap = {c.lower(): c for c in df.columns}
    # Normalize column names
    rename = {}
    for key in ["store_menu", "date", "pred", "test_id"]:
        if key in lcmap: rename[lcmap[key]] = key
    # pred_d1..pred_d7
    for i in range(1, H+1):
        key = f"pred_d{i}"
        for c in df.columns:
            if c.lower() == key:
                rename[c] = key
    if rename:
        df = df.rename(columns=rename)
    return df

def assemble_from_saved(saved_files):
    frames = []
    for fp in saved_files:
        df = load_flexible_pred(fp)
        if "store_menu" not in df.columns:
            continue
        # Try to infer TEST id
        base = os.path.basename(fp)
        test_id = None
        for k in tests.keys():
            if k in base:
                test_id = k
                break
        if test_id is None and "test_id" in df.columns:
            test_id = df["test_id"].iloc[0]
        if test_id is None:
            # fallback to first TEST
            test_id = sorted(tests.keys())[0]
        # Convert to long form with actual future dates
        if {"date", "pred"}.issubset(df.columns):
            tmp = df.copy()
            tmp["test_id"] = test_id
            frames.append(tmp[["test_id", "store_menu", "date", "pred"]])
        elif all([f"pred_d{i}" in df.columns for i in range(1, H+1)]):
            recs = []
            for _, row in df.iterrows():
                sm = row["store_menu"]
                fdates = future_dates_map[test_id]
                for i, d in enumerate(fdates, start=1):
                    recs.append((test_id, sm, d, row[f"pred_d{i}"]))
            tmp = pd.DataFrame(recs, columns=["test_id","store_menu","date","pred"])
            frames.append(tmp)
        else:
            print(f"Skip unrecognized schema: {fp}")
    if not frames:
        return None
    all_long = pd.concat(frames, ignore_index=True)
    return all_long

assembled = assemble_from_saved(saved_pred_files)
if assembled is not None:
    # Pivot to submission shape: rows=dates, cols=store_menu
    assembled["date"] = pd.to_datetime(assembled["date"]).dt.strftime("%Y-%m-%d")
    # Ensure coverage: for each TEST id and store_menu there should be 7 rows
    coverage = assembled.groupby(["test_id","store_menu"]).size().describe()
    print("Per-series row count stats:\n", coverage)

    submission = []
    for test_id in sorted(tests.keys()):
        subset = assembled[assembled["test_id"]==test_id]
        # Build full grid to avoid missing combos
        sms = tests[test_id]["store_menu"].unique()
        grid = pd.MultiIndex.from_product([future_dates_map[test_id], sms], names=["date","store_menu"]).to_frame(index=False)
        merged = grid.merge(subset, on=["date","store_menu"], how="left")
        # If missing predictions, fill with 0
        merged["pred"] = merged["pred"].fillna(0.0)
        wide = merged.pivot(index="date", columns="store_menu", values="pred").sort_index()
        submission.append(wide)

    submission_df = pd.concat(submission, axis=0).sort_index()
    out_path = os.path.join(RESULT_DIR, "submission_from_saved_preds.csv")
    submission_df.to_csv(out_path, index=True)
    print(f"Saved: {out_path}  shape={submission_df.shape}")
else:
    print("No saved TEST predictions detected. You can run Mode B (direct inference) or fallback baseline below.")


In [ ]:

# %% [Fallback] Naive baseline (repeat last value) -> builds a valid submission shape
def naive_predict_last_value(test_df):
    # test_df contains 28 days per store_menu
    preds = []
    for sm, g in test_df.groupby("store_menu"):
        last_val = max(0.0, float(g.sort_values("date")["sales"].iloc[-1]))
        preds.append((sm, [last_val]*H))
    out = pd.DataFrame(preds, columns=["store_menu", "preds"])
    for i in range(1, H+1):
        out[f"pred_d{i}"] = out["preds"].apply(lambda x: x[i-1])
    return out.drop(columns=["preds"])

baseline_blocks = []
for test_id, df in tests.items():
    block = naive_predict_last_value(df)
    # explode to dates
    recs = []
    fdates = future_dates_map[test_id]
    for _, row in block.iterrows():
        for i, d in enumerate(fdates, start=1):
            recs.append((test_id, row["store_menu"], d, row[f"pred_d{i}"]))
    baseline_blocks.append(pd.DataFrame(recs, columns=["test_id","store_menu","date","pred"]))

baseline_long = pd.concat(baseline_blocks, ignore_index=True)
baseline_long["date"] = pd.to_datetime(baseline_long["date"]).dt.strftime("%Y-%m-%d")

# pivot
submission_baseline = []
for test_id in sorted(tests.keys()):
    subset = baseline_long[baseline_long["test_id"]==test_id]
    wide = subset.pivot(index="date", columns="store_menu", values="pred").sort_index()
    submission_baseline.append(wide)

submission_baseline_df = pd.concat(submission_baseline, axis=0).sort_index()
baseline_path = os.path.join(RESULT_DIR, "submission_naive_last.csv")
submission_baseline_df.to_csv(baseline_path)
print(f"Saved baseline submission: {baseline_path}  shape={submission_baseline_df.shape}")


In [ ]:

# %% [Mode B] Direct inference with PatchTST if checkpoint + config are available
# This block tries to reconstruct the model from config and run forward on TEST_*.
# If your training notebook exported modules to ./artifacts_patchtst/notebook_cells, imports will succeed.

import importlib, os, math
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def find_config_and_ckpt(art_dir=ARTIFACT_DIR):
    if not os.path.isdir(art_dir):
        return None, None
    cfgs = sorted(glob.glob(os.path.join(art_dir, "config_*.json")))
    ckpts = sorted(glob.glob(os.path.join(art_dir, "*_best.ckpt")))
    return (cfgs[-1] if cfgs else None), (ckpts[-1] if ckpts else None)

cfg_path, ckpt_path = find_config_and_ckpt()
print("config:", cfg_path)
print("ckpt  :", ckpt_path)

# Try to import existing model implementation if available
def try_import_patchtst_module():
    cand_dirs = [
        os.path.join(ARTIFACT_DIR, "notebook_cells"),
        ARTIFACT_DIR,
        ".",
    ]
    for d in cand_dirs:
        if not os.path.isdir(d): 
            continue
        sys.path.insert(0, d)
        for py in glob.glob(os.path.join(d, "*.py")):
            if "patch" in os.path.basename(py).lower() or "tst" in os.path.basename(py).lower():
                mod_name = os.path.splitext(os.path.basename(py))[0]
                try:
                    mod = importlib.import_module(mod_name)
                    for name in dir(mod):
                        if name.lower().startswith("patchtst"):
                            cls = getattr(mod, name)
                            if isinstance(cls, type):
                                return mod, cls
                except Exception as e:
                    continue
    return None, None

mod, PatchTSTClass = try_import_patchtst_module()

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=2048):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        n = x.size(1)
        return x + self.pe[:, :n, :]

class SimplePatchTST(nn.Module):
    # Channel-independent PatchTST for inference only.
    # Input: (B, C, L) -> Output: (B, H) for sales channel index 0
    def __init__(self, C, L=28, H=7, patch_len=7, stride=1, d_model=128, nhead=8, num_layers=3, dropout=0.1):
        super().__init__()
        assert L >= patch_len, "L must be >= patch_len"
        self.C, self.L, self.H = C, L, H
        self.patch_len = patch_len
        self.stride = stride
        self.N = 1 + (L - patch_len) // stride
        self.proj = nn.Linear(patch_len, d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dropout=dropout, batch_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.pos = PositionalEncoding(d_model, max_len=self.N)
        self.head = nn.Linear(d_model*self.N, H)
        self.instance_norm = True

    def _patchify(self, x):
        B, C, L = x.shape
        patches = []
        for s in range(0, L - self.patch_len + 1, self.stride):
            patches.append(x[:, :, s:s+self.patch_len].unsqueeze(2))
        x_p = torch.cat(patches, dim=2)  # (B, C, N, P)
        x_p = x_p.reshape(B*C, x_p.size(2), x_p.size(3))  # (B*C, N, P)
        return x_p

    def forward(self, x):
        B, C, L = x.shape
        assert C == self.C and L == self.L
        if self.instance_norm:
            mean = x.mean(dim=-1, keepdim=True)
            std = x.std(dim=-1, keepdim=True).clamp_min(1e-6)
            x = (x - mean) / std
        x_p = self._patchify(x)
        z = self.proj(x_p)
        z = self.pos(z)
        z = self.encoder(z)
        z = z.reshape(B, C, z.size(1)*z.size(2))
        y = self.head(z[:, 0, :])
        return y

def load_config_defaults(path):
    cfg = {
        "L": L, "H": H, "patch_len": 7, "stride": 1,
        "d_model": 128, "nhead": 8, "num_layers": 3, "dropout": 0.1,
        "inputs": "all",
        "columns": None,
    }
    if path and os.path.exists(path):
        try:
            with open(path, "r", encoding="utf-8") as f:
                j = json.load(f)
            for k in cfg.keys():
                if k in j: cfg[k] = j[k]
        except Exception as e:
            print("config parse failed, use defaults:", e)
    return cfg

cfg = load_config_defaults(cfg_path)
print("Resolved config:", cfg)

ALL_COLS = ["sales","date_ordinal"]
INPUT_USE_ALL = (cfg.get("inputs","all") != "sales")
if INPUT_USE_ALL:
    sample_test = next(iter(tests.values()))
    numeric_cols = [c for c in sample_test.columns if pd.api.types.is_numeric_dtype(sample_test[c])]
    feature_cols = list(dict.fromkeys([c for c in numeric_cols]))
else:
    feature_cols = ["sales"]

if "sales" in feature_cols:
    feature_cols = ["sales"] + [c for c in feature_cols if c!="sales"]

print("Feature cols:", feature_cols)

C = len(feature_cols)
if PatchTSTClass is not None:
    try:
        model = PatchTSTClass(C=C, L=cfg["L"], H=cfg["H"], patch_len=cfg["patch_len"], stride=cfg["stride"],
                              d_model=cfg["d_model"], nhead=cfg["nhead"], num_layers=cfg["num_layers"], dropout=cfg["dropout"])
    except TypeError:
        model = SimplePatchTST(C=C, L=cfg["L"], H=cfg["H"], patch_len=cfg["patch_len"], stride=cfg["stride"],
                               d_model=cfg["d_model"], nhead=cfg["nhead"], num_layers=cfg["num_layers"], dropout=cfg["dropout"])
else:
    model = SimplePatchTST(C=C, L=cfg["L"], H=cfg["H"], patch_len=cfg["patch_len"], stride=cfg["stride"],
                           d_model=cfg["d_model"], nhead=cfg["nhead"], num_layers=cfg["num_layers"], dropout=cfg["dropout"])

if ckpt_path and os.path.exists(ckpt_path):
    state = torch.load(ckpt_path, map_location="cpu")
    sd = None
    if isinstance(state, dict):
        if "model_state_dict" in state: sd = state["model_state_dict"]
        elif "state_dict" in state: sd = state["state_dict"]
        else: sd = state
    try:
        model.load_state_dict(sd, strict=False)
        print("Loaded checkpoint with strict=False")
    except Exception as e:
        print("Checkpoint load failed:", repr(e))
else:
    print("No checkpoint found. The model will produce random-ish outputs. Prefer Mode A or baseline.")

model = model.to(DEVICE)
model.eval()

# Fit per-column normalization on train (mean/std), numeric only
num_cols_in_train = [c for c in train.columns if pd.api.types.is_numeric_dtype(train[c]) and c in feature_cols]
train_stats = train[num_cols_in_train].agg(["mean","std"]).to_dict()
for k in list(train_stats.keys()):
    if np.isnan(train_stats[k]["std"]) or train_stats[k]["std"] < 1e-6:
        train_stats[k]["std"] = 1.0

def make_model_input(test_df, sm, feature_cols, L):
    g = test_df[test_df["store_menu"]==sm].sort_values("date").tail(L)
    assert len(g) == L, f"{sm} needs {L} steps, got {len(g)}"
    X = []
    for col in feature_cols:
        v = g[col].astype(float).values
        if col in train_stats:
            m = train_stats[col]["mean"]; s = train_stats[col]["std"]
            if s <= 0: s = 1.0
            v = (v - m)/s
        X.append(v)
    x = np.stack(X, axis=0)
    return x

pred_blocks = []
for test_id, df in tests.items():
    sms = df["store_menu"].unique()
    X_batch = []
    sm_index = []
    for sm in sms:
        x = make_model_input(df, sm, feature_cols, L=L)
        X_batch.append(x)
        sm_index.append(sm)
    X = torch.tensor(np.stack(X_batch, axis=0), dtype=torch.float32, device=DEVICE)
    with torch.no_grad():
        y = model(X)
    preds = y.detach().cpu().numpy()
    if "sales" in train_stats:
        m = train_stats["sales"]["mean"]; s = train_stats["sales"]["std"]
        preds = preds * s + m
    preds = np.clip(preds, 0.0, None)
    recs = []
    fdates = future_dates_map[test_id]
    for sm, row in zip(sm_index, preds):
        for i, d in enumerate(fdates):
            recs.append((test_id, sm, d, float(row[i])))
    pred_blocks.append(pd.DataFrame(recs, columns=["test_id","store_menu","date","pred"]))

pred_long = pd.concat(pred_blocks, ignore_index=True)
pred_long["date"] = pd.to_datetime(pred_long["date"]).dt.strftime("%Y-%m-%d")

submission_m = []
for test_id in sorted(tests.keys()):
    subset = pred_long[pred_long["test_id"]==test_id]
    wide = subset.pivot(index="date", columns="store_menu", values="pred").sort_index()
    submission_m.append(wide)
submission_m_df = pd.concat(submission_m, axis=0).sort_index()
out_direct = os.path.join(RESULT_DIR, "submission_patchtst.csv")
submission_m_df.to_csv(out_direct)
print(f"Saved PatchTST inference submission: {out_direct}  shape={submission_m_df.shape}")


In [ ]:

# %% [Optional] LightGBM-style target feature builder (for traditional models)
# Generates future 7-day feature rows per (store_menu), using last 28-day window for rolling features.
import pandas as pd
import numpy as np

def build_calendar_feats(dates):
    dt = pd.to_datetime(dates)
    return pd.DataFrame({
        "date": dt.strftime("%Y-%m-%d"),
        "date_ordinal": dt.map(lambda x: x.toordinal()).astype(int),
        "dow": dt.dayofweek.astype(int),
        "dom": dt.day.astype(int),
        "month": dt.month.astype(int),
        "weekofyear": dt.isocalendar().week.astype(int),
        "is_weekend": (dt.dayofweek >= 5).astype(int),
    })

def build_target_aux_features(tests_dict, L=28, H=7):
    out = {}
    for test_id, df in tests_dict.items():
        feats = []
        last_date = pd.to_datetime(df['date']).max().strftime('%Y-%m-%d')
        fdates = pd.to_datetime([(pd.to_datetime(last_date) + pd.Timedelta(days=i)).strftime('%Y-%m-%d') for i in range(1, H+1)])
        cal = build_calendar_feats(fdates)
        by = df.groupby("store_menu")
        for sm, g in by:
            g = g.sort_values("date").tail(L)
            sales_vals = g["sales"].values.astype(float)
            last = float(sales_vals[-1])
            roll7 = float(pd.Series(sales_vals).tail(7).mean())
            roll14 = float(pd.Series(sales_vals).tail(14).mean())
            roll28 = float(pd.Series(sales_vals).tail(28).mean()) if len(sales_vals)>=28 else float(pd.Series(sales_vals).mean())
            tmp = cal.copy()
            tmp["store_menu"] = sm
            tmp["last"] = last
            tmp["roll7"] = roll7
            tmp["roll14"] = roll14
            tmp["roll28"] = roll28
            feats.append(tmp)
        out[test_id] = pd.concat(feats, ignore_index=True)
    return out

target_aux = build_target_aux_features(tests, L=L, H=H)
k0 = sorted(target_aux.keys())[0]
print("Aux features example:", k0)
display(target_aux[k0].head())


In [ ]:

# %% [Done] Notebook info
print("Notebook finished. See files saved under:", RESULT_DIR)
